# AutoDrive Multi-Agent Training Notebook

**Two agents × 10 Indian road checkpoints × sudden mid-task alerts**

This notebook:
1. Starts the AutoDrive server in the background
2. Runs both agents together through the 10-checkpoint city route
3. Handles sudden alerts (ambulance, pothole, cattle, flash crowd, …) fired mid-task
4. Shows a live reward curve after every N episodes
5. Prints only meaningful output: failures, rewards, sudden alerts, checkpoint progress

**You do NOT need two separate terminals.** Both agents run in the same notebook loop.

## 1 — Install dependencies

In [ ]:
# Run once if packages are missing
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "requests", "matplotlib", "ipywidgets", "tqdm"], check=True)
print('Dependencies ready.')

## 2 — Start the server

In [ ]:
import subprocess, time, sys, os

SERVER_PORT = 8000
BASE_URL    = f"http://localhost:{SERVER_PORT}"

# ------------------------------------------------------------------
# Change this path if the notebook lives elsewhere
# ------------------------------------------------------------------
REPO_DIR = os.path.abspath(os.path.join(os.path.dirname(os.path.abspath('__file__')), '.'))
print('Repo dir:', REPO_DIR)

_server_proc = subprocess.Popen(
    [sys.executable, "-m", "autodrive_openenv.server.app"],
    cwd=os.path.dirname(REPO_DIR),   # parent of the package dir
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

# Wait until server is ready
import requests
for _ in range(20):
    time.sleep(1)
    try:
        r = requests.get(f"{BASE_URL}/healthz", timeout=2)
        if r.status_code == 200:
            print('Server ready ✓', r.json())
            break
    except Exception:
        pass
else:
    raise RuntimeError('Server did not start. Check the process output.')

## 3 — Agent decision function

Replace the heuristic with any LLM call, or keep the rule-based fallback for fast local testing.

In [ ]:
import json

# ── Heuristic agent (no LLM needed) ──────────────────────────────────────────
# Reacts to sudden alerts and nearby hazards with correct rule-based responses.

AMBULANCE_ACTIONS  = ["steer_left",  0.7]
POTHOLE_ACTIONS    = ["steer_right", 0.6]
CATTLE_ACTIONS     = ["brake",       0.9]
CROWD_ACTIONS      = ["brake",       0.8]
BRIDGE_ACTIONS     = ["change_lane_left", 0.6]
WRONG_WAY_ACTIONS  = ["brake",       1.0]
DEFAULT_ACTION     = ["accelerate",  0.4]
BRAKE_HARD         = ["brake",       0.85]

_ALERT_RESPONSE = {
    "ambulance":    AMBULANCE_ACTIONS,
    "pothole":      POTHOLE_ACTIONS,
    "cattle":       CATTLE_ACTIONS,
    "crowd":        CROWD_ACTIONS,
    "bridge":       BRIDGE_ACTIONS,
    "wrong_way":    WRONG_WAY_ACTIONS,
    "speed_breaker": ["brake", 0.7],
    "water":        ["brake", 0.6],
}

def decide(vehicle_id: str, obs: dict, sudden_alert: dict) -> dict:
    """Heuristic decision function — replace inner logic with an LLM call if desired."""
    action, value = DEFAULT_ACTION

    # React to sudden alert fired THIS STEP
    if sudden_alert:
        msg = sudden_alert.get("message", "").lower()
        for keyword, resp in _ALERT_RESPONSE.items():
            if keyword in msg:
                action, value = resp
                break
        else:
            action, value = BRAKE_HARD  # unknown alert → brake hard to be safe

    else:
        # Normal driving logic
        hazard_dist  = obs.get("nearest_hazard_m", 999)
        near_miss    = obs.get("near_miss", False)
        inbox        = obs.get("negotiation_inbox", [])

        # Check inbox for signals from the other vehicle
        for msg in inbox:
            if "braking" in msg.lower() or "hazard" in msg.lower():
                action, value = "brake", 0.6
                break
            if "clear" in msg.lower() or "accelerating" in msg.lower():
                action, value = "accelerate", 0.5
                break

        if action == "accelerate":   # not overridden by inbox
            if hazard_dist < 8:
                action, value = "brake", 0.9
            elif hazard_dist < 15:
                action, value = "brake", 0.6
            elif near_miss:
                action, value = "steer_left", 0.5

    return {"action": action, "value": round(float(value), 2)}

print('Heuristic agent ready.')

## 4 — Training loop (both agents, 10 checkpoints, sudden alerts)

In [ ]:
import requests, json, time, math
from collections import defaultdict

# ── Config ────────────────────────────────────────────────────────────────────
N_EPISODES       = 20      # episodes to train (increase for real training)
PLOT_EVERY       = 5       # show reward curve every N episodes
MAX_STEPS_GUARD  = 400     # hard cap per episode (10 CPs × ~36 max steps each)
N_VEHICLES       = 2
VEHICLE_IDS      = [f"vehicle_{i}" for i in range(N_VEHICLES)]

# ── Tracking ──────────────────────────────────────────────────────────────────
ep_rewards          = []        # total reward per episode
ep_cp_cleared       = []        # checkpoints cleared per episode
ep_sudden_alerts    = []        # sudden alerts handled per episode
ep_failures         = []        # checkpoint failures per episode
ep_weaknesses       = defaultdict(int)   # scenario_type → failure count


# ── Helpers ───────────────────────────────────────────────────────────────────
def post(endpoint, **kwargs):
    return requests.post(f"{BASE_URL}{endpoint}", **kwargs, timeout=10).json()

def get(endpoint):
    return requests.get(f"{BASE_URL}{endpoint}", timeout=10).json()


print(f"Starting training: {N_EPISODES} episodes, {N_VEHICLES} agents, 10-checkpoint city route")
print("=" * 70)

for ep in range(1, N_EPISODES + 1):
    # ── Reset fleet ───────────────────────────────────────────────────────────
    reset_r = post("/fleet/reset",
                   json={"n_vehicles": N_VEHICLES, "continue_on_fail": True})
    if reset_r.get("status") != "ok":
        print(f"[Ep {ep}] Reset failed: {reset_r}")
        continue

    ep_total_reward  = 0.0
    ep_alerts_count  = 0
    ep_cp_done       = 0
    ep_fail_count    = 0
    step             = 0
    last_obs         = {vid: {} for vid in VEHICLE_IDS}

    while step < MAX_STEPS_GUARD:
        step += 1

        # ── Both agents propose actions ────────────────────────────────────────
        # sudden_alert comes from the LAST step's result; agents react this step
        actions = {}
        for vid in VEHICLE_IDS:
            obs = last_obs.get(vid, {})
            # sudden_alert is shared across fleet — same alert for all agents
            actions[vid] = decide(vid, obs, obs.get("sudden_alert"))

        # ── Step all agents simultaneously ────────────────────────────────────
        result = post("/fleet/step_all", json=actions)
        if result.get("status") != "ok":
            print(f"[Ep {ep}] Step error: {result.get('error')}")
            break

        # ── Process results ────────────────────────────────────────────────────
        sudden_alert  = result.get("sudden_alert")
        route_event   = result.get("route_event")
        route_state   = result.get("route_state", {})
        v_obs         = result.get("vehicle_observations", {})

        # Forward observations to next step
        for vid in VEHICLE_IDS:
            if vid in v_obs and not v_obs[vid].get("skipped"):
                last_obs[vid] = {**v_obs[vid], "sudden_alert": sudden_alert}

        # Accumulate rewards
        for vid in VEHICLE_IDS:
            ep_total_reward += v_obs.get(vid, {}).get("reward", 0.0)

        # Sudden alert
        if sudden_alert:
            ep_alerts_count += 1
            print(f"  [Ep {ep} | Step {step}] {sudden_alert['message']}")

        # Checkpoint transitions
        if route_event:
            ev_type = route_event.get("type", "")
            if ev_type == "checkpoint_cleared":
                ep_cp_done += 1
                cp_info = route_event.get("next_checkpoint") or {}
                print(
                    f"  [Ep {ep} | Step {step}] ✓ CHECKPOINT CLEARED by {route_event['cleared_by']} "
                    f"(+{route_event['reward']:.2f}) → next: {cp_info.get('name', 'DESTINATION')}"
                )
            elif ev_type == "checkpoint_failed":
                ep_fail_count += 1
                cp = route_state.get("current_checkpoint", {})
                cp_name = cp.get("name", "?")
                scenario = cp.get("scenario_type", "?")
                print(
                    f"  [Ep {ep} | Step {step}] ✗ CHECKPOINT FAILED: {cp_name} "
                    f"(penalty {route_event['penalty']:.1f}) [scenario: {scenario}]"
                )
                ep_weaknesses[scenario] += 1

        if result.get("fleet_done"):
            if route_state.get("route_completed"):
                ep_total_reward += 20.0  # route completion bonus
            break

    ep_rewards.append(round(ep_total_reward, 2))
    ep_cp_cleared.append(ep_cp_done)
    ep_sudden_alerts.append(ep_alerts_count)
    ep_failures.append(ep_fail_count)

    # ── Per-episode summary ────────────────────────────────────────────────────
    pct = route_state.get("progress_pct", 0.0)
    print(
        f"[Ep {ep:>3}] reward={ep_total_reward:+.2f}  "
        f"CPs_cleared={ep_cp_done}/10  "
        f"failures={ep_fail_count}  "
        f"sudden_alerts={ep_alerts_count}  "
        f"route_progress={pct*100:.0f}%  "
        f"steps={step}"
    )

    # ── Periodic reward curve ──────────────────────────────────────────────────
    if ep % PLOT_EVERY == 0:
        import matplotlib.pyplot as plt
        import matplotlib.ticker as ticker

        fig, axes = plt.subplots(2, 2, figsize=(13, 7))
        fig.suptitle(f'AutoDrive Fleet Training — After Episode {ep}', fontsize=14, fontweight='bold')

        xs = list(range(1, len(ep_rewards) + 1))
        roll = lambda lst, w: [
            sum(lst[max(0,i-w):i])/len(lst[max(0,i-w):i]) for i in range(1, len(lst)+1)
        ]

        # Reward curve
        ax = axes[0, 0]
        ax.plot(xs, ep_rewards, color='#4C9BE8', alpha=0.4, linewidth=1, label='Per-episode')
        ax.plot(xs, roll(ep_rewards, 5), color='#1A5FAD', linewidth=2, label='5-ep rolling avg')
        ax.axhline(0, color='gray', linewidth=0.8, linestyle='--')
        ax.set_title('Total Reward per Episode')
        ax.set_xlabel('Episode')
        ax.set_ylabel('Reward')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

        # Checkpoints cleared
        ax = axes[0, 1]
        ax.bar(xs, ep_cp_cleared, color='#3DAA6B', alpha=0.75)
        ax.plot(xs, roll(ep_cp_cleared, 5), color='#1A6B3D', linewidth=2)
        ax.set_ylim(0, 10.5)
        ax.set_title('Checkpoints Cleared per Episode')
        ax.set_xlabel('Episode')
        ax.set_ylabel('Checkpoints (out of 10)')
        ax.yaxis.set_major_locator(ticker.MultipleLocator(2))
        ax.grid(True, alpha=0.3)

        # Failures vs sudden alerts
        ax = axes[1, 0]
        ax.plot(xs, ep_failures, color='#E84C4C', linewidth=1.5, marker='x', label='Failures')
        ax.plot(xs, ep_sudden_alerts, color='#E8A84C', linewidth=1.5, marker='o', label='Sudden Alerts')
        ax.set_title('Failures vs Sudden Alerts per Episode')
        ax.set_xlabel('Episode')
        ax.set_ylabel('Count')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

        # Weakness heatmap (failure by scenario type)
        ax = axes[1, 1]
        if ep_weaknesses:
            scenarios = sorted(ep_weaknesses, key=ep_weaknesses.get, reverse=True)
            counts    = [ep_weaknesses[s] for s in scenarios]
            colors    = ['#E84C4C' if c == max(counts) else '#E8A84C' for c in counts]
            ax.barh(scenarios, counts, color=colors)
            ax.set_title('Weakness Map (failures by scenario)')
            ax.set_xlabel('Failure count')
            ax.invert_yaxis()
        else:
            ax.text(0.5, 0.5, 'No failures yet!', ha='center', va='center',
                    fontsize=12, color='green', transform=ax.transAxes)
            ax.set_title('Weakness Map')
        ax.grid(True, alpha=0.3, axis='x')

        plt.tight_layout()
        plt.savefig(f'reward_curves_ep{ep}.png', dpi=120, bbox_inches='tight')
        plt.show()
        print(f'  → Reward curve saved to reward_curves_ep{ep}.png')

print("\n" + "=" * 70)
print("TRAINING COMPLETE")
print(f"  Best episode reward : {max(ep_rewards):.2f}  (ep {ep_rewards.index(max(ep_rewards))+1})")
print(f"  Avg reward (all)    : {sum(ep_rewards)/len(ep_rewards):.2f}")
print(f"  Total sudden alerts : {sum(ep_sudden_alerts)}")
print(f"  Total failures      : {sum(ep_failures)}")
if ep_weaknesses:
    worst = max(ep_weaknesses, key=ep_weaknesses.get)
    print(f"  Biggest weakness    : '{worst}' ({ep_weaknesses[worst]} failures)")
print("=" * 70)

## 5 — Final reward curve

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('AutoDrive — Final Training Summary', fontsize=14, fontweight='bold')

xs   = list(range(1, len(ep_rewards) + 1))
roll = lambda lst, w: [
    sum(lst[max(0,i-w):i])/len(lst[max(0,i-w):i]) for i in range(1, len(lst)+1)
]

# Reward
ax = axes[0]
ax.fill_between(xs, ep_rewards, alpha=0.2, color='#4C9BE8')
ax.plot(xs, ep_rewards, color='#4C9BE8', linewidth=1, alpha=0.6)
ax.plot(xs, roll(ep_rewards, 5), color='#1A5FAD', linewidth=2.5, label='Rolling avg')
ax.set_title('Cumulative Reward'); ax.set_xlabel('Episode'); ax.set_ylabel('Reward')
ax.legend(); ax.grid(True, alpha=0.3)

# CPs cleared
ax = axes[1]
ax.bar(xs, ep_cp_cleared, color='#3DAA6B', alpha=0.7)
ax.plot(xs, roll(ep_cp_cleared, 5), color='#1A5FAD', linewidth=2)
ax.set_ylim(0, 10.5)
ax.set_title('Checkpoints Cleared / Episode')
ax.set_xlabel('Episode'); ax.set_ylabel('Count (max 10)')
ax.grid(True, alpha=0.3)

# Weakness bar
ax = axes[2]
if ep_weaknesses:
    scenarios = sorted(ep_weaknesses, key=ep_weaknesses.get, reverse=True)[:8]
    counts    = [ep_weaknesses[s] for s in scenarios]
    ax.barh(scenarios, counts, color='#E84C4C')
    ax.set_title('Agent Weaknesses (failures by scenario)')
    ax.set_xlabel('Failures')
    ax.invert_yaxis()
else:
    ax.text(0.5, 0.5, 'No failures!', ha='center', va='center',
            fontsize=14, color='green', transform=ax.transAxes)
    ax.set_title('Agent Weaknesses')
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('reward_curves_final.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to reward_curves_final.png')

## 6 — Stop server

In [ ]:
_server_proc.terminate()
print('Server stopped.')